<a href="https://colab.research.google.com/github/thehmfpk/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit
**Lane:** same as ML-04 through ML-08 — `fact_content_daily_performance`, month `2026-03`, `is_declining_label` target.

Load `hunting-leakage-and-validating` + `flyrank/flyrank-data` per `skills/README.md` if working with an assistant.


## 1. Two paper findings + my methodology questions

**FILL IN BEFORE SUBMITTING** — the two findings below are placeholders. Replace each with a real finding from *FlyRank Research: The State of AI-Driven SEO in Numbers*, paraphrased in your own words (don't paste the paper's exact sentences). Keep the tone constructive — the goal is practicing the question, not grading the paper.

### Finding #1
> *[Paraphrase the first finding here, e.g. "Content refreshed with AI-assisted rewrites saw a reported increase in organic clicks over N weeks."]*

**My methodology question:** Where does the underlying metric come from — is "organic clicks" measured directly from GSC/analytics, or self-reported by survey participants? If it's a before/after comparison, is there a control group of content that wasn't refreshed, or could the same seasonal/trend effects that caused the refresh decision also explain the lift on their own (regression to the mean, or reviewers picking already-recovering content to refresh)?

### Finding #2
> *[Paraphrase the second finding here, e.g. "X% of sites using AI-driven optimization reported improved rankings."]*

**My methodology question:** Does the validation design support a causal claim, or only a correlational one? If this is drawn from a survey of self-selected respondents, how might non-response or selection bias (sites that adopted the tool because they were already investing more in SEO) inflate the reported effect versus a randomized or matched comparison?


## 0. Setup — reload March 2026 + rebuild the ML-08 feature vector, label, and baseline

In [1]:
import pandas as pd, numpy as np, os, sklearn
from datasets import load_dataset
from huggingface_hub import HfApi
from google.colab import userdata

print(f'scikit-learn version: {sklearn.__version__}')
RANDOM_SEED = 42

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

api = HfApi(token=HF_TOKEN)
all_files = api.list_repo_files('FlyRank/internship-warehouse', repo_type='dataset')
march_files = [f for f in all_files if 'fact_content_daily_performance' in f
               and 'sample' not in f and '2026-03' in f]
if not march_files:
    print('[WARNING] No files matched - inspect all_files and fix the filter:')
    for f in [x for x in all_files if 'fact_content_daily_performance' in x and 'sample' not in x][:20]:
        print(' ', f)
    raise ValueError('Fix march_files filter above, then re-run.')

dataset = load_dataset('FlyRank/internship-warehouse', data_files={'train': march_files}, split='train')
needed_cols = ['report_date','client_hash_id','content_hash_id',
               'gsc_data_available','gsc_impressions','gsc_clicks','gsc_avg_position']
cols_present = [c for c in needed_cols if c in dataset.column_names]
dataset = dataset.select_columns(cols_present)
raw = dataset.to_pandas()
raw['report_date'] = pd.to_datetime(raw['report_date'])
raw['day_of_month'] = raw['report_date'].dt.day
raw = raw[raw['gsc_data_available'] == True].copy()
print(f'Loaded {len(raw)} GSC-available rows for March 2026.')


scikit-learn version: 1.6.1


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 3611061 GSC-available rows for March 2026.


In [2]:
first_half = raw[raw['day_of_month'] <= 15]
second_half = raw[raw['day_of_month'] > 15]

feat = (first_half.groupby(['client_hash_id','content_hash_id'])
        .agg(clicks_first_half=('gsc_clicks','sum'),
             impressions_first_half=('gsc_impressions','sum'),
             avg_position_first_half=('gsc_avg_position', lambda s: s[s > 0].mean()),
             active_days_first_half=('gsc_impressions', lambda s: (s > 0).sum()))
        .reset_index())
feat = feat[feat['impressions_first_half'] > 0].copy()
feat['ctr_first_half'] = feat['clicks_first_half'] / feat['impressions_first_half']
feat['has_position_data'] = feat['avg_position_first_half'].notna().astype(int)
feat['avg_position_first_half'] = feat['avg_position_first_half'].fillna(feat['avg_position_first_half'].median())

position_bins = [0, 3, 6, 10, 20, 50, np.inf]
position_labels = ['1-3','4-6','7-10','11-20','21-50','51+']
feat['position_bucket_fh'] = pd.cut(feat['avg_position_first_half'], bins=position_bins, labels=position_labels)
position_dummies = pd.get_dummies(feat['position_bucket_fh'], prefix='pos_bucket')

feature_vector = pd.concat([
    feat[['client_hash_id','content_hash_id','clicks_first_half','impressions_first_half',
          'ctr_first_half','avg_position_first_half','has_position_data','active_days_first_half']],
    position_dummies
], axis=1)

label_df = (second_half.groupby(['client_hash_id','content_hash_id'])
            .agg(clicks_second_half=('gsc_clicks','sum')).reset_index())

data = feature_vector.merge(label_df, on=['client_hash_id','content_hash_id'], how='inner')
data['is_declining_label'] = (data['clicks_second_half'] < data['clicks_first_half']).astype(int)
data = data.dropna()
honest_feature_cols = [c for c in feature_vector.columns if c not in ('client_hash_id','content_hash_id')]
print(f'{len(data)} rows. Base rate (declining share): {data["is_declining_label"].mean():.3f}')


141467 rows. Base rate (declining share): 0.203


## 2. My model under an honest split (before/after)

**Before:** a plain random row split (no grouping) — the kind of split that looks fine but lets the model partially memorize same-client siblings. **After:** the grouped-by-client split ML-08 actually used. Same features, same model type, same metrics — only the split changes.

In [3]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

X = data[honest_feature_cols]
y = data['is_declining_label']

def precision_at_k(df, score_col, label_col, k):
    return df.nlargest(k, score_col)[label_col].mean()

K_VALUES = [20, 50, 100]

# --- BEFORE: naive random split, no grouping ---
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y)
rf_random = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=RANDOM_SEED, n_jobs=-1)
rf_random.fit(Xr_train, yr_train)
random_test_df = data.loc[Xr_test.index].copy()
random_test_df['score'] = rf_random.predict_proba(Xr_test)[:, 1]

# --- AFTER: grouped split by client_hash_id (the honest one, same as ML-08) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(data, groups=data['client_hash_id']))
Xg_train, Xg_test = X.iloc[train_idx], X.iloc[test_idx]
yg_train, yg_test = y.iloc[train_idx], y.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=RANDOM_SEED, n_jobs=-1)
rf_grouped.fit(Xg_train, yg_train)
grouped_test_df = data.iloc[test_idx].copy()
grouped_test_df['score'] = rf_grouped.predict_proba(Xg_test)[:, 1]

overlap = set(data.iloc[train_idx]['client_hash_id']) & set(data.iloc[test_idx]['client_hash_id'])
print(f'Grouped split - clients overlapping train/test: {len(overlap)} (must be 0)')
assert len(overlap) == 0


Grouped split - clients overlapping train/test: 0 (must be 0)


In [4]:
before_after_rows = []
for name, df, label_col in [('BEFORE - random split', random_test_df, 'is_declining_label'),
                             ('AFTER - grouped split', grouped_test_df, 'is_declining_label')]:
    row = {'split': name, 'base_rate': df[label_col].mean()}
    for k in K_VALUES:
        row[f'precision@{k}'] = precision_at_k(df, 'score', label_col, k)
    before_after_rows.append(row)

before_after_table = pd.DataFrame(before_after_rows)
print('--- Before/after: same model, only the split changed ---')
print(before_after_table.to_string(index=False))

gap = before_after_table.iloc[0][[c for c in before_after_table.columns if 'precision' in c]] - \
      before_after_table.iloc[1][[c for c in before_after_table.columns if 'precision' in c]]
print('\nGap (random-split precision minus grouped-split precision) at each K:')
print(gap.to_string())
print('\nA positive gap here is the memorization signal the grouped split is supposed to catch -')
print('same-client sibling content in both train and random-test lets the model partially learn')
print('client-specific patterns rather than the general signal.')


--- Before/after: same model, only the split changed ---
                split  base_rate  precision@20  precision@50  precision@100
BEFORE - random split   0.202647          0.80          0.88           0.83
AFTER - grouped split   0.252035          0.85          0.90           0.87

Gap (random-split precision minus grouped-split precision) at each K:
precision@20    -0.05
precision@50    -0.02
precision@100   -0.04

A positive gap here is the memorization signal the grouped split is supposed to catch -
same-client sibling content in both train and random-test lets the model partially learn
client-specific patterns rather than the general signal.


## 3. Leakage audit

Same attack-checklist hunt as ML-04/05, run again on this notebook's final feature set.

In [5]:
print('--- Attack checklist ---')

# 1. Timeline: every feature from days 1-15, label strictly from days 16-31
print('[1] Timeline: features from day_of_month<=15, label from day_of_month>15 -',
      'confirmed by construction in section 0/1 above.')

# 2. No label-derived or sibling columns in features
suspicious_cols = [c for c in honest_feature_cols if 'second' in c.lower() or 'label' in c.lower()]
print(f'[2] Label-derived/sibling columns in features: {suspicious_cols if suspicious_cols else "none"}')

# 3. No product flags / existing-system scores as features
flag_cols = [c for c in honest_feature_cols if 'flag' in c.lower() or 'score' in c.lower()]
print(f'[3] Product-flag / existing-system-score columns in features: {flag_cols if flag_cols else "none"}')

# 4. Grouped split used
print(f'[4] Grouped split used in section 2, zero client overlap confirmed: yes')

# 5. Base rate printed next to every metric - done in section 2 table
print(f'[5] Base rate printed alongside precision@K: yes (see before/after table columns)')

# 6+7. The deliberate-leak test - add clicks_second_half, watch the score jump, then remove it
leaky_X_train = data.iloc[train_idx][honest_feature_cols + ['clicks_second_half']]
leaky_X_test = data.iloc[test_idx][honest_feature_cols + ['clicks_second_half']]
rf_leaky = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=RANDOM_SEED, n_jobs=-1)
rf_leaky.fit(leaky_X_train, yg_train)
leaky_test_df = data.iloc[test_idx].copy()
leaky_test_df['score'] = rf_leaky.predict_proba(leaky_X_test)[:, 1]

honest_p50 = precision_at_k(grouped_test_df, 'score', 'is_declining_label', 50)
leaky_p50 = precision_at_k(leaky_test_df, 'score', 'is_declining_label', 50)
print(f"\n[6] Deliberate-leak test at precision@50: honest={honest_p50:.3f}, "
      f"with clicks_second_half added={leaky_p50:.3f}")
if leaky_p50 > honest_p50:
    print('    Confirms the test harness detects leakage correctly (score jumped as expected).')
    print('    clicks_second_half is NOT in the shipped feature set - this was only run to verify the harness.')
else:
    print('    [WARNING] Score did not jump - re-check the harness itself before trusting any other result here.')

# 7. Metrics recomputed out-of-fold (the grouped test set was never seen during rf_grouped.fit)
print('[7] All metrics above computed on held-out test indices never seen during training: yes')


--- Attack checklist ---
[1] Timeline: features from day_of_month<=15, label from day_of_month>15 - confirmed by construction in section 0/1 above.
[2] Label-derived/sibling columns in features: none
[3] Product-flag / existing-system-score columns in features: none
[4] Grouped split used in section 2, zero client overlap confirmed: yes
[5] Base rate printed alongside precision@K: yes (see before/after table columns)

[6] Deliberate-leak test at precision@50: honest=0.900, with clicks_second_half added=1.000
    Confirms the test harness detects leakage correctly (score jumped as expected).
    clicks_second_half is NOT in the shipped feature set - this was only run to verify the harness.
[7] All metrics above computed on held-out test indices never seen during training: yes


## 4. Claim rewrite

**Original (bold) claim, from ML-08's submission note:**
> *"Random Forest beats the Week-4 baseline at precision@50 for predicting content decline."*

**Rewritten in safe language:**
> On a client-grouped, held-out March 2026 split, the Random Forest model showed an **observed** precision@50 of [fill in your actual number] versus the Week-4 rule baseline's [fill in], and versus a base rate of [fill in] — a **directional**, single-month result intended for **decision-support** prioritization, not a **measured** causal claim about what drives decline, and not yet validated across other months or client segments.
